# Majority Voting for Annotations

This notebook performs majority voting across multiple model annotations. It reads corresponding CSV files from three directories representing different models, compares the annotations row-by-row, and outputs a final set of annotations based on the majority vote.

In cases where there is a 3-way tie (i.e., all three models produce a unique label), the system flags the annotation as `Needs Human Review`.

In [ ]:
import os
import pandas as pd
from pathlib import Path
from collections import Counter

### Helper Function
This function calculates the majority vote from a list of votes. It automatically handles three-way ties.

In [ ]:
def get_majority_vote(votes):
    """
    Returns the majority vote from a list of votes. 
    If there is a tie (e.g., all models give unique labels, or no valid votes), 
    we flag it as 'Needs Human Review'.
    """
    # Remove NaN values and empty strings to ensure we are voting on actual labels
    valid_votes = [v for v in votes if pd.notna(v) and str(v).strip() != ""]
    if not valid_votes:
        return "Needs Human Review"
    
    counts = Counter(valid_votes)
    most_common = counts.most_common(2)
    
    # If there is a tie for the most common label
    if len(most_common) > 1 and most_common[0][1] == most_common[1][1]:
        return "Needs Human Review"
        
    return most_common[0][0]

### Configure Data Paths
Enter the absolute or relative paths to your three dataset directories. If you are running this in Google Colab or Kaggle, make sure to mount your drive or upload your folders, and then provide the respective paths when prompted.

In [ ]:
print("Please provide the paths to the folders containing the CSV files for each model.")
gpt_path_input = input("Path for GPT-5.2 data: ").strip()
gemini_path_input = input("Path for Gemini 3.5 flash data: ").strip()
sonet_path_input = input("Path for Sonet 4.6 data: ").strip()
output_path_input = input("Path to save the output Majority Voting data: ").strip()

# Convert the string inputs to Path objects for easy manipulation
gpt_dir = Path(gpt_path_input)
gemini_dir = Path(gemini_path_input)
sonet_dir = Path(sonet_path_input)
output_dir = Path(output_path_input)

model_dirs = {
    "GPT-5.2": gpt_dir,
    "Gemini 3.5 flash": gemini_dir,
    "Sonet 4.6": sonet_dir
}

print("\n--- Setup Complete ---")
print(f"Output Directory set to: {output_dir}")

### Execution
This cell loops through all CSV files in the GPT directory, reads the corresponding files from Gemini and Sonet, performs the majority voting row-by-row, and saves the final outputs into your specified output folder.

In [ ]:
reference_model_name = "GPT-5.2"
reference_model_dir = model_dirs[reference_model_name]

if not reference_model_dir.exists():
    print(f"Error: Reference directory '{reference_model_dir}' does not exist. Please check your path.")
else:
    # Iterate through all CSV files in the reference directory
    for file_path in reference_model_dir.rglob("*.csv"):
        rel_path = file_path.relative_to(reference_model_dir)
        
        # Load dataframes for all 3 models
        dfs = []
        for model_name, m_dir in model_dirs.items():
            model_file = m_dir / rel_path
            if model_file.exists():
                try:
                    dfs.append(pd.read_csv(model_file))
                except Exception as e:
                    print(f"Error reading {model_file}: {e}")
            else:
                print(f"Warning: {model_file} not found for model {model_name}.")
                
        if len(dfs) < 2:
            print(f"Not enough models found for {rel_path}. Skipping.")
            continue
            
        # Validate row counts across all files
        min_len = min(len(df) for df in dfs)
        if any(len(df) != min_len for df in dfs):
            print(f"Warning: Row counts mismatch for {rel_path}. Using minimum row count {min_len}.")
            
        # Base our final output entirely on the first dataset to ensure structure matching
        final_df = dfs[0].copy().head(min_len)
        annotation_cols = [col for col in final_df.columns if col != 'comment']
        
        # Loop through each row and column, calculating the vote
        for idx in range(min_len):
            for col in annotation_cols:
                votes = [df.at[idx, col] for df in dfs if col in df.columns]
                final_df.at[idx, col] = get_majority_vote(votes)
                    
        # Prepare output directory
        out_file = output_dir / rel_path
        out_file.parent.mkdir(parents=True, exist_ok=True)
        
        # Save and log
        final_df.to_csv(out_file, index=False)
        print(f"Processed and saved: {out_file}")
        
    print("\n✅ Majority voting complete! Results have been saved.")